In [22]:
import rectanglepy as rectangle
import pandas as pd
from anndata import AnnData
import h5py
import spatialdata_io
import helper
import biomart

In [ ]:


counts = pd.read_csv('../data/human_m1/matrix.csv', index_col=0)
counts = counts.astype(int)

metadata = pd.read_csv('../data/human_m1/metadata.csv', index_col=0)


In [3]:
annotations = metadata['cell_type_alias_label']
# remove anything after the second space(word)
annotations = annotations.str.split(' ', n=2).str[:2].str.join(' ')
# remove anything afert first minus
annotations = annotations.str.split('-', n=1).str[0]

In [4]:
adata = AnnData(counts, obs=annotations.to_frame(name='cell_type'))

In [3]:
f = spatialdata_io.visium(path='../data', dataset_id='151673',scalefactors_file='scalefactors_json.json', tissue_positions_file='tissue_positions.csv', counts_file='151673_filtered_feature_bc_matrix.h5')


INFO     reading ../data/151673_filtered_feature_bc_matrix.h5                                                      


/Users/berni/PycharmProjects/rectangle_paper/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/berni/PycharmProjects/rectangle_paper/.venv/lib/python3.12/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/berni/PycharmProjects/rectangle_paper/.venv/lib/python3.12/site-packages/spatialdata/models/models.py:1144: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)


In [20]:
data_table = f['table']
bulks = data_table.to_df()

In [12]:
gene_ids = data_table.var['gene_ids']


In [23]:

server = biomart.BiomartServer('http://uswest.ensembl.org/biomart')
mart = server.datasets['mmusculus_gene_ensembl']
attributes = ['ensembl_transcript_id', 'mgi_symbol',
              'ensembl_gene_id', 'transcript_length']
response = mart.search({'attributes': attributes, 'filters':{'ensembl_gene_id': gene_ids}})
data = response.raw.data.decode('ascii')



HTTPError: 503 Server Error: Service Temporarily Unavailable for url: http://uswest.ensembl.org/biomart/martservice

In [ ]:
# Calculate TPM properly: normalize by gene length first, then by library size
# TPM = (count / gene_length) / sum(count / gene_length) * 1e6
rpk = bulks.div(gene_lengths, axis=1)  # Reads per kilobase
scaling_factor = rpk.sum(axis=1)  # Sum of RPK values per sample
bulks_tpm = rpk.div(scaling_factor, axis=0) * 1e6  # TPM calculation


In [ ]:
estimations, signature_result = rectangle.rectangle(adata, bulks_tpm)